<a href="https://colab.research.google.com/github/JacobSong1/MindWatch-IE08/blob/backend-dev/EDA/notebooks/MW_201_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from google.colab import userdata
from pathlib import Path
import subprocess
import os

GITHUB_USERNAME = "bakai-work07"
REPO_URL = "https://github.com/JacobSong1/MindWatch-IE08.git"
REPO_DIR = Path("/content/MindWatch-IE08")
BASE_BRANCH = "backend-dev"

token = userdata.get("GITHUB_TOKEN")

if not token:
    raise ValueError("GITHUB_TOKEN was not found in Colab Secrets.")

# Keep credentials only in this temporary Colab runtime
subprocess.run(
    [
        "git", "config", "--global",
        "credential.helper", "cache --timeout=21600"
    ],
    check=True
)

credential = (
    "protocol=https\n"
    "host=github.com\n"
    f"username={GITHUB_USERNAME}\n"
    f"password={token}\n\n"
)

subprocess.run(
    ["git", "credential", "approve"],
    input=credential,
    text=True,
    check=True
)

del token, credential

if (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin"],
        check=True
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "switch", BASE_BRANCH],
        check=True
    )
    subprocess.run(
        [
            "git", "-C", str(REPO_DIR),
            "pull", "--ff-only", "origin", BASE_BRANCH
        ],
        check=True
    )
    print("Existing repository updated.")
else:
    subprocess.run(
        [
            "git", "clone",
            "--branch", BASE_BRANCH,
            REPO_URL,
            str(REPO_DIR)
        ],
        check=True
    )
    print("Repository cloned.")

EDA_DIR = REPO_DIR / "EDA"
os.chdir(EDA_DIR)

print("Working directory:", Path.cwd())

subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["git", "status", "--short"], check=True)

Repository cloned.
Working directory: /content/MindWatch-IE08/EDA


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [6]:
!git status
!git branch --show-current
!ls

On branch backend-dev
Your branch is up to date with 'origin/backend-dev'.

nothing to commit, working tree clean
backend-dev
data  notebooks  README.md  requirements.txt


# MW-201: Exploratory Data Analysis

## Objective

This notebook examines wearable-derived physiological measures and
self-reported stress for one student. It evaluates data quality, temporal
patterns, relationships between measures, and potential physiological–
perceived stress mismatch.

In [11]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = Path(
    "/content/drive/MyDrive/MindWatch/SSAQS"
)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_ROOT}")

participant_dirs = sorted(
    [
        path for path in DATA_ROOT.iterdir()
        if path.is_dir() and path.name.isdigit()
    ],
    key=lambda path: int(path.name)
)

print("Participant folders:", len(participant_dirs))
print("First folders:", [p.name for p in participant_dirs[:35]])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Participant folders: 35
First folders: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35']


In [12]:
import pandas as pd

EXPECTED_FILES = [
    "activity_level.csv",
    "daily_questions.csv",
    "hrv.csv",
    "oxygen.csv",
    "sleep.csv",
    "steps.csv",
    "stress.csv",
]

inventory_rows = []

for participant_dir in participant_dirs:
    for filename in EXPECTED_FILES:
        path = participant_dir / filename

        if path.exists():
            df = pd.read_csv(path)

            inventory_rows.append({
                "participant_id": participant_dir.name,
                "file": filename,
                "exists": True,
                "rows": len(df),
                "columns": len(df.columns),
                "duplicate_rows": int(df.duplicated().sum()),
            })
        else:
            inventory_rows.append({
                "participant_id": participant_dir.name,
                "file": filename,
                "exists": False,
                "rows": 0,
                "columns": 0,
                "duplicate_rows": None,
            })

inventory = pd.DataFrame(inventory_rows)

display(
    inventory.groupby("file").agg(
        participants_present=("exists", "sum"),
        total_rows=("rows", "sum"),
        median_rows=("rows", "median"),
        duplicate_rows=("duplicate_rows", "sum"),
    )
)

,participants_present,total_rows,median_rows,duplicate_rows
file,,,,
activity_level.csv,32,1766244,48506.0,0.0
daily_questions.csv,35,3133,103.0,0.0
hrv.csv,32,151243,4375.0,0.0
oxygen.csv,32,978367,27540.0,0.0
sleep.csv,32,1829,49.0,0.0
steps.csv,32,658894,18029.0,0.0
stress.csv,32,1777,49.0,0.0


In [13]:
print((DATA_ROOT / "course-details.csv").exists())
print((DATA_ROOT / "users-courses.csv").exists())

True
True


In [14]:
for filename in EXPECTED_FILES:
    examples = [
        participant_dir / filename
        for participant_dir in participant_dirs
        if (participant_dir / filename).exists()
    ]

    if not examples:
        print(f"\n{filename}: NOT FOUND")
        continue

    sample = pd.read_csv(examples[0])

    time_columns = [
        column for column in sample.columns
        if "time" in column.lower() or "date" in column.lower()
    ]

    print(f"\n--- {filename} ---")
    print("Columns:", sample.columns.tolist())
    print("Dtypes:")
    print(sample.dtypes.astype(str).to_dict())

    if time_columns:
        print("Timestamp samples:")
        print(sample[time_columns].head(3).to_dict("records"))


--- activity_level.csv ---
Columns: ['timestamp', 'level']
Dtypes:
{'timestamp': 'object', 'level': 'object'}
Timestamp samples:
[{'timestamp': '2025-02-21T12:58:00Z'}, {'timestamp': '2025-02-21T12:59:00Z'}, {'timestamp': '2025-02-21T13:00:00Z'}]

--- daily_questions.csv ---
Columns: ['timeStampScheduled', 'timeStampSent', 'timeStampStart', 'timeStampStop', 'originalTimeStampSent', 'timeZoneOffset', 'deltaUTC', 'stress', 'anxiety']
Dtypes:
{'timeStampScheduled': 'int64', 'timeStampSent': 'int64', 'timeStampStart': 'int64', 'timeStampStop': 'int64', 'originalTimeStampSent': 'int64', 'timeZoneOffset': 'int64', 'deltaUTC': 'int64', 'stress': 'int64', 'anxiety': 'int64'}
Timestamp samples:
[{'timeStampScheduled': 1739999335, 'timeStampSent': 1739999336, 'timeStampStart': 1740000118, 'timeStampStop': 1740000146, 'originalTimeStampSent': 0, 'timeZoneOffset': -21600}, {'timeStampScheduled': 1740082074, 'timeStampSent': 1740082078, 'timeStampStart': 1740082084, 'timeStampStop': 1740082096, 'o

In [15]:
def load_participant_file(filename):
    frames = []

    for participant_dir in participant_dirs:
        path = participant_dir / filename

        if not path.exists():
            continue

        df = pd.read_csv(path)
        df.insert(0, "participant_id", participant_dir.name)
        frames.append(df)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


questions = load_participant_file("daily_questions.csv")
hrv = load_participant_file("hrv.csv")
sleep = load_participant_file("sleep.csv")
steps = load_participant_file("steps.csv")
activity = load_participant_file("activity_level.csv")
fitbit_stress = load_participant_file("stress.csv")
oxygen = load_participant_file("oxygen.csv")